In [11]:
# Cell 1 — Load and Validate Cleaned Data
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

df = pd.read_csv("../data/processed/customer_support_tickets_clean.csv")

print(f"Duplicated tickets: {df.duplicated().sum()}")

invalid_type = df["Ticket Type"].fillna("").str.strip().eq("").sum()
print(f"Invalid or empty Ticket Type values: {invalid_type}")

invalid_description = (
    df["Ticket Description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Invalid or empty Ticket Description values: {invalid_description}")

Duplicated tickets: 0
Invalid or empty Ticket Type values: 0
Invalid or empty Ticket Description values: 0


In [12]:
# Cell 2 — Define X and Y
X = df["Ticket Description"]
y = df["Ticket Type"]

In [13]:
# Cell 3 — Create 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state= 42
)

In [14]:
# Cell 4 — Create Vectorizer
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [15]:
# Cell 5 — Create and train Logistic Regression Model
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [16]:
# Cell 6 — Define classify_ticket() Function
def classify_ticket(ticket_description):
    if ticket_description.strip() == "":
        return "Unable to classify"

    ticket_tfidf = vectorizer.transform([ticket_description])
    prediction = model.predict(ticket_tfidf)

    return prediction[0]

In [17]:
# Cell 7 — Test Classifier with One Ticket
new_ticket_description = (
    "My computer cannot load data after I click the load button. "
    "How can I fix it? Thanks!"
)

prediction = classify_ticket(new_ticket_description)

print("Predicted Ticket Type:", prediction)

Predicted Ticket Type: Product Inquiry


In [18]:
# Cell 8 — Test Classifier with Multiple Example Tickets
descriptions = [
    "My account shows the payment for anti-virus system was made twice while I was attempting several times. When am I expected to receive my money back?",
    "I want to verify why my computer freezes while I run my browser and does not save the last edits in my reports.",
    "My browser does not log in after I submit the right passcode. Can you verify? Thanks.",
    "I would like to cancel the plan for this data cleaning service."
]

predictions = [
    classify_ticket(description)
    for description in descriptions
]

example_results = pd.DataFrame({
    "Ticket Description": descriptions,
    "Predicted Ticket Type": predictions
})

example_results

,Ticket Description,Predicted Ticket Type
0,My account shows the payment for anti-virus sy...,Technical Issue
1,I want to verify why my computer freezes while...,Cancellation Request
2,My browser does not log in after I submit the ...,Refund Request
3,I would like to cancel the plan for this data ...,Refund Request


In [19]:
# Cell 9 — Test Invalid or Edge-Case Inputs
invalids = [
    "",
    " ",
    "       ",
    "\t"
]

failed_predictions = [
    classify_ticket(invalid)
    for invalid in invalids
]

print(failed_predictions)

['Unable to classify', 'Unable to classify', 'Unable to classify', 'Unable to classify']


In [20]:
# Cell 10 — Inspect Prediction Confidence

confidence_ticket = descriptions[0]

confidence_tfidf = vectorizer.transform([confidence_ticket])

probabilities = model.predict_proba(confidence_tfidf)[0]

confidence_results = pd.DataFrame({
    "Ticket Type": model.classes_,
    "Probability": probabilities
})

confidence_results = confidence_results.sort_values(
    by="Probability",
    ascending=False
)

confidence_results

,Ticket Type,Probability
4,Technical Issue,0.401971
1,Cancellation Request,0.194433
0,Billing Inquiry,0.167443
2,Product Inquiry,0.132403
3,Refund Request,0.103749


## Day 27 Conclusions

### Reusable Ticket Classification Function

A reusable `classify_ticket()` function was created to classify one new customer-support ticket at a time. The function:

1. Checks whether the ticket description is empty or contains only whitespace.
2. Uses the previously fitted TF-IDF vectorizer to transform the new description into the same feature space used during training.
3. Uses the trained Logistic Regression model to predict the Ticket Type.
4. Returns the single predicted Ticket Type.

This allows the trained ML pipeline to classify new ticket descriptions without retraining the vectorizer or classifier.

### Testing New Ticket Descriptions

The classifier successfully generated predictions for manually written ticket descriptions. However, several predictions did not intuitively correspond to the content of the examples.

For example:

- A computer loading problem was classified as **Product Inquiry**.
- A ticket describing a duplicated payment and asking about receiving money back was classified as **Technical Issue**.
- A ticket explicitly asking to cancel a service was classified as **Refund Request**.

These results are consistent with the weak model performance observed during Day 26 and show that a functioning classification pipeline does not necessarily mean that the classifier is reliable.

### Input Validation

The function was tested with:

- An empty string
- A single space
- Multiple spaces
- A tab character

All four inputs correctly returned **"Unable to classify"**.

This prevents empty or whitespace-only descriptions from being passed to the classifier as meaningful tickets.

### Prediction Probabilities

Prediction probabilities were inspected for the first manually written example. The model assigned the following probabilities:

| Ticket Type | Probability |
| --- | ---: |
| Technical Issue | 40.20% |
| Cancellation Request | 19.44% |
| Billing Inquiry | 16.74% |
| Product Inquiry | 13.24% |
| Refund Request | 10.37% |

The model therefore selected **Technical Issue**, which had the highest estimated probability. However, the probability was only about 40%, while substantial probability remained distributed across the other Ticket Types.

These probabilities represent the fitted Logistic Regression model's estimated class probabilities and should not be interpreted as guaranteed real-world confidence.

### Conclusion

Day 27 converted the TF-IDF and Logistic Regression work from the previous days into a reusable classification function that can accept new ticket descriptions and return predictions.

The technical pipeline now works from end to end:

**New Ticket → Input Validation → TF-IDF Transformation → Logistic Regression → Ticket Type Prediction**

However, the manually written test cases reinforce the low predictive performance identified on Day 26. The classifier is therefore useful as a working **traditional ML baseline**, but the current model should not be treated as a reliable production ticket classifier.

This baseline can now be compared with the LLM-based ticket analysis approach in the next stage of the project.